In [ ]:
import os, glob
import torch
import matplotlib.pyplot as plt

from monai.inferers import sliding_window_inference
from monai.transforms import SaveImaged

os.environ["CUDA_VISIBLE_DEVICES"] = "5"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)

In [29]:
# load best model for testing
best_path = os.path.join("outputs/real+uncond3","unet3d_last_*.pt")
best_path = glob.glob(best_path)[0]

ckpt = torch.load(best_path, map_location=device)
model.load_state_dict(ckpt["model"])
print("Loaded:", best_path)


Loaded: outputs/real+uncond3/unet3d_last_15.pt


In [ ]:
from monai.data import decollate_batch
dice_metric = DiceMetric(include_background=False, reduction="mean")
model.eval()

pred_save = SaveImaged(
    keys="pred",
    meta_keys="image_meta_dict",
    output_dir=OUT_DIR,
    output_postfix="seg",
    output_ext=".nii.gz",
    resample=False,
)

with torch.no_grad():
    for i, batch in enumerate(val_loader):
        if i != 7:
            continue
        y = batch["image"].to(device)
        x = batch["label"].to(device)
        logits = sliding_window_inference(y, ROI_SIZE, SW_BATCH_SIZE, model)  # (B,1,H,W,D)

        x_pred = post_trans(logits) # normalize and binary
        
        batch["pred"] = x_pred
        
        dice = dice_metric(x_pred, x)
        dice_val = dice.item()

        print(f"Case {i+1} dice: {dice_val:.4f}")
        # print(np.percentile(y, [1,5,10]))

        y = y.squeeze().cpu().numpy()
        x = x.squeeze().cpu().numpy()
        x_pred = x_pred.squeeze().cpu().numpy()

        # plt.figure(figsize=(6,4))
        # plt.hist(y.flatten(), bins=200)

        z = 80
        plt.figure(figsize=(12,4))
        plt.subplot(1,3,1)
        plt.imshow(y[:,:,z], cmap="gray")
        plt.title("Image")
        plt.axis("off")

        plt.subplot(1,3,2)
        plt.imshow(x[:,:,z], cmap="gray")
        plt.title("GT Label")
        plt.axis("off")

        plt.subplot(1,3,3)
        plt.imshow(x_pred[:,:,z], cmap="gray")
        plt.title("Pred Mask")
        plt.axis("off")
        plt.suptitle(f"Mid slice z={z}")
        plt.show()

        # plt.figure(figsize=(6,6))
        # plt.imshow(y[:,:,z], cmap="gray")
        # plt.imshow(x[:,:,z], alpha=0.35)
        # plt.imshow(x_pred[:,:,z], alpha=0.35)
        # plt.title("Overlay (Image + Pred)")
        # plt.axis("off")
        # plt.show()
        
        # for b in decollate_batch(batch):
        #     pred_save(b)

print("Saved predictions to:", OUT_DIR)
